# Training Pipeline — CO-QCNN (U1)

A **Multi-Channel Quantum Convolutional Neural Network** (Channel Overwrite), ported from
[`anthonysmaldone/QCNN-Multi-Channel-Supervised-Learning`](https://github.com/anthonysmaldone/QCNN-Multi-Channel-Supervised-Learning)
(Smaldone & Batista, *Hardware-adaptable quantum circuits for convolutional layers with
multi-channel data*, Quantum Machine Intelligence, 2023,
[doi:10.1007/s42484-023-00130-3](https://doi.org/10.1007/s42484-023-00130-3)). Like
[`qcnn.ipynb`](./qcnn.ipynb) and [`quonv.ipynb`](./quonv.ipynb), there is no
PennyLane/TensorFlow-Quantum dependency here — the load-bearing circuit is ported as plain `torch`
tensor ops in [`cnn/models/mcqcnn.py`](../../models/mcqcnn.py).

Trained and evaluated on **BreastMNIST**, matching every other notebook in this folder — same
dataset, same [`_handlers/evaluation.py::evaluate_all_metrics`](../../../_handlers/evaluation.py)
metric suite, so all the quantum and classical runs are directly comparable.

**This notebook is one of ten.** `train.py`'s menu offers ten trainable variants — five
architectures (CO, PCO, PCO-T, WEV, control) x two circuit ansatzes (U1, U2). This folder has one
notebook each, and they are **identical except for the single factory call in the _Model_ cell**.

## What all ten share

A genuine sliding **2x2, stride-1, no-padding quantum convolution** (`extract_patches`), with
`n_kernels=3` independently-learned parameter sets acting as three output feature maps — the
quantum analogue of a classical `Conv2D`. Each patch's 4 pixels are `RX` angle-encoded onto 4
qubits, an entangling schedule of **fractional powers of X and Z** (`CXPowGate`/`CZPowGate`,
continuously interpolating identity <-> CNOT/CZ via a trainable exponent) is applied, and one
qubit is measured. Every circuit then applies the same `arccos(clip(e)) / pi` un-embedding before
the classical head — `Flatten` -> `Dense(32, relu)` -> `Dense(n_classes)`, exactly `models.py`.

The ten differ **only in how the channels are combined**, which is the paper's entire subject.

## This variant: CO-QCNN (U1)

`registers=1`, no inter-register entangler. A **single** 4-qubit data register is reused
across all channels: the encoder runs `circuit_layers = ceil(n_channels / registers)` = 3 sequential
waves, and each wave's `RX` angle encoding writes *over* the register state the previous channel
left behind (hence "overwrite"), with the entangling schedule applied after each wave. No two
channels are ever resident in the circuit at the same time, so channel mixing is purely sequential.

**1 ancilla + 4 data = 5 qubits.** The cheapest of the ten to simulate.

**U1 ansatz** (`circuits.py::U1_circuit`) — the intra-channel entangler is a 4-cycle of `CXPowGate`: `1->0`, `2->1`, `3->2`, `0->3`.

## The other nine

| Notebook | Variant | Circuit | Qubits |
|---|---|---|---|
| [`mcqcnn_co_u1.ipynb`](./mcqcnn_co_u1.ipynb) | CO-QCNN (U1) | `U1Circuit(registers=1)` | 5 |
| [`mcqcnn_pco_u1.ipynb`](./mcqcnn_pco_u1.ipynb) | PCO-QCNN (U1) | `U1Circuit(registers=3, rdpa=3, inter_U=True)` | 13 |
| [`mcqcnn_pco_t_u1.ipynb`](./mcqcnn_pco_t_u1.ipynb) | PCO-T-QCNN (U1) | `U1Circuit(registers=3, rdpa=1, inter_U=True)` | 15 |
| [`mcqcnn_wev_u1.ipynb`](./mcqcnn_wev_u1.ipynb) | WEV-QCNN (U1) | `QU1Control(classical_weights=True)` | 4 |
| [`mcqcnn_control_u1.ipynb`](./mcqcnn_control_u1.ipynb) | Control QCNN (U1) | `QU1Control()` | 4 |
| [`mcqcnn_co_u2.ipynb`](./mcqcnn_co_u2.ipynb) | CO-QCNN (U2) | `U2Circuit(registers=1)` | 5 |
| [`mcqcnn_pco_u2.ipynb`](./mcqcnn_pco_u2.ipynb) | PCO-QCNN (U2) | `U2Circuit(registers=3, rdpa=3, inter_U=True)` | 13 |
| [`mcqcnn_pco_t_u2.ipynb`](./mcqcnn_pco_t_u2.ipynb) | PCO-T-QCNN (U2) | `U2Circuit(registers=3, rdpa=1, inter_U=True)` | 15 |
| [`mcqcnn_wev_u2.ipynb`](./mcqcnn_wev_u2.ipynb) | WEV-QCNN (U2) | `QU2Control(classical_weights=True)` | 4 |
| [`mcqcnn_control_u2.ipynb`](./mcqcnn_control_u2.ipynb) | Control QCNN (U2) | `QU2Control()` | 4 |

## Note on the channel count

BreastMNIST is grayscale, and the MC-QCNN variants only differ from each other in how they combine
channels — a 1-channel input would collapse all ten into near-identical models. The dataset cell
therefore replicates the single channel into 3, the same grayscale->RGB adaptation
[`qtransfer.ipynb`](./qtransfer.ipynb) makes for ResNet18, mapping onto the original's 3-channel
`COLORS`/`CIFAR10` path (`registers=3`). **The three channels carry identical data**, so the
multi-channel *machinery* is exercised but the inter-channel *information* the paper's ansatzes
are built to capture is not there to find — expect the five architectures to separate much less
than they do in the paper. Point `args.dataset` at a natively-RGB MedMNIST flag (`bloodmnist`,
`pathmnist`, `dermamnist`, `retinamnist`) for a run where the channels actually differ.

## Install Requirements

What the pipeline imports: `torch` for the model/training loop, `medmnist` for BreastMNIST,
`scikit-learn` for the metrics, plus `tqdm`. No `tensorflow`, `tensorflow-quantum` or `cirq` — the
circuits are reimplemented as dense `torch` tensor ops, so the original's quantum-simulation stack
is never installed.

In [1]:
!nvidia-smi

Tue Jul 28 14:55:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.2/908.2 MB 1.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 92.2 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 82.8 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 68.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 88.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.

In [3]:
!pip install medmnist==3.0.2 scikit-learn tqdm requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 3.8 MB/s eta 0:00:00


## Get the survey code

The model and handlers live in this repository's `survey/src/cnn` and `survey/src/_handlers`
packages, so the notebook needs a checkout of it. On Colab it clones into
`/content/quantum-quantization`, or `git pull --ff-only`s that directory if it is already there —
so re-running the cell after a push picks up the new code. Run locally, the notebook already sits
inside the repo, so `find_src` climbs to `survey/src` and git is never touched (your working tree
is left alone).

The branch is chosen by *environment*, not by working directory: the imports cell `os.chdir`s into
the checkout, so a cwd-based test would find `cnn` on every re-run and silently skip the pull.

If the repository is private the anonymous clone fails with an authentication error; use a token
URL instead — `REPO_URL = 'https://<GITHUB_TOKEN>@github.com/alexandrachirita98/quantum-quantization.git'`.

In [4]:
import pathlib
import subprocess
import sys

REPO_URL = 'https://github.com/alexandrachirita98/quantum-quantization.git'

# NB: key off the environment, not the working directory. `os.chdir` in the imports cell moves the
# cwd *inside* the checkout, so a cwd-based test would report "already have the code" on every
# re-run and silently skip the pull.
IN_COLAB = 'google.colab' in sys.modules or pathlib.Path('/content').is_dir()
CLONE_DIR = pathlib.Path('/content/quantum-quantization')   # where the Colab checkout goes


def find_src(start):
    """Climb from `start` looking for the survey `src/` root — the directory holding `cnn`."""
    for p in [pathlib.Path(start), *pathlib.Path(start).parents]:
        if (p / 'cnn' / 'models' / 'mcqcnn.py').is_file():
            return p
    return None


if IN_COLAB:
    if CLONE_DIR.exists():                                  # refresh whatever was cloned earlier
        subprocess.run(['git', 'pull', '--ff-only'], cwd=str(CLONE_DIR), check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(CLONE_DIR)], check=True)
    SRC = CLONE_DIR / 'survey' / 'src'
else:                                                       # local: the notebook lives in the repo
    SRC = find_src(pathlib.Path.cwd())

assert SRC is not None and (SRC / 'cnn' / 'models' / 'mcqcnn.py').is_file(), f'mcqcnn.py not found under {SRC}'
print('survey src:', SRC)

Cloning into '/content/quantum-quantization'...


survey src: /content/quantum-quantization/survey/src


## Imports

`CO_U1_QCNN` (this notebook's variant) and the training helpers come from the survey's `cnn`
package; `evaluate_all_metrics` (the same all-metrics evaluator every classical notebook uses)
comes from `_handlers`. `SRC` from the previous cell goes on `sys.path`, and `os.chdir` moves into
it so `./data` (the shared [`src/data`](../../../data) folder) is where `medmnist` downloads
BreastMNIST.

In [5]:
import os
import sys

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from medmnist import BreastMNIST

# import the pipeline from the survey `src/` root, and work from there so './data' resolves inside it
sys.path.insert(0, str(SRC))
os.chdir(SRC)
print('working directory:', os.getcwd())

# drop cached modules, so a `git pull` above is actually reflected on a re-run
for _m in [m for m in list(sys.modules) if m in ('cnn', '_handlers') or m.startswith(('cnn.', '_handlers.'))]:
    del sys.modules[_m]

from cnn.models.mcqcnn import CO_U1_QCNN
from cnn.handlers.mcqcnn import (
    train_mcqcnn,
    evaluate,
    resize_images,
    RawImageDataset,
    MCQCNNLogits,
)
from _handlers.evaluation import evaluate_all_metrics

working directory: /content/quantum-quantization/survey/src


## Configuration

`train.py`'s shipped defaults, kept as a plain namespace instead of its interactive menu:
`lr=0.001` (`datamenu2`'s default), Adam, sparse-categorical cross-entropy.

**Image size** — `10x10`. Every model in `models.py` declares
`tf.keras.layers.Input((10,10,C))`, so BreastMNIST's native 28x28 is bilinear-resized down to
match. This is not cosmetic: the quantum conv slides over every patch, and 28x28 would mean 729
patches per image instead of 81.

**Dataset** — `breastmnist`, the same medmnist flag every other notebook in this folder trains on.

**Batch size and epochs deviate from the original** (`50` and `20`). The heavier variants simulate
13- and 15-qubit statevectors for `batch_size * 81 * 3` patches at once, with autograd retaining an
intermediate per gate — at batch 50 that is tens of gigabytes. `batch_size=4` keeps all ten
runnable from one shared config; `epochs=5` keeps the demo run bounded, the same reduction
[`quonv.ipynb`](./quonv.ipynb) makes. Raise both for the 4- and 5-qubit variants (CO, WEV,
control), where memory is not a constraint.

In [6]:
from types import SimpleNamespace

args = SimpleNamespace(
    dataset='breastmnist',   # a medmnist flag, native 28x28 resolution
    img_size=10,             # models.py hardcodes Input((10,10,C))
    n_channels=3,            # grayscale replicated to 3, matching the original's COLORS path
    n_classes=2,             # BreastMNIST is binary (malignant vs. normal/benign)
    lr=0.001,                # train.py's default learning rate
    batch_size=4,            # original uses 50; lowered so the 15-qubit variants fit in memory
    epochs=5,                # original uses 20; kept small for a demo run
)

## Device

In [7]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using {} device.".format(device))

Using cuda:0 device.


## Dataset

BreastMNIST (malignant vs. normal/benign, already binary). Each native 28x28 image is
bilinear-resized down to `img_size x img_size`, kept in `[0, 1]`, replicated across `n_channels`
and returned **channel-last** as `(B, H, W, C)` — the TensorFlow layout the ported models expect,
not PyTorch's usual `(B, C, H, W)`.

`train_raw`/`test_raw` also back a `RawImageDataset` each, used only by the final evaluation cell
below — `evaluate_all_metrics` expects a `Dataset` of raw `(image, label)` pairs and applies the
resize itself (via `MCQCNNLogits`), the same way `quonv.ipynb` hands it raw images plus a logits
adapter.

In [8]:
os.makedirs('./data', exist_ok=True)   # BreastMNIST() checks root exists *before* downloading

train_raw = BreastMNIST(split='train', download=True, root='./data')
test_raw = BreastMNIST(split='test', download=True, root='./data')

train_x = resize_images(train_raw.imgs, args.img_size, args.n_channels)
train_y = torch.from_numpy(train_raw.labels).reshape(-1).long()
test_x = resize_images(test_raw.imgs, args.img_size, args.n_channels)
test_y = torch.from_numpy(test_raw.labels).reshape(-1).long()

train_loader = data.DataLoader(data.TensorDataset(train_x, train_y), batch_size=args.batch_size, shuffle=True)
test_loader = data.DataLoader(data.TensorDataset(test_x, test_y), batch_size=args.batch_size, shuffle=False)

train_dataset = RawImageDataset(train_raw.imgs, train_raw.labels)
test_dataset = RawImageDataset(test_raw.imgs, test_raw.labels)

print('train:', train_x.shape, train_y.shape)   # (N, 10, 10, 3) channel-last
print('test: ', test_x.shape, test_y.shape)

100%|██████████| 560k/560k [00:00<00:00, 736kB/s] 

Using downloaded and verified file: ./data/breastmnist.npz
train: torch.Size([546, 10, 10, 3]) torch.Size([546])
test:  torch.Size([156, 10, 10, 3]) torch.Size([156])


## Model

**This is the only cell that differs between the ten notebooks.**

`CO_U1_QCNN` is this repository's port of `models.py::Channel Overwrite` — it builds the quantum-conv layer
described at the top, then `Flatten` -> `Dense(32, relu)` -> `Dense(n_classes)`. The softmax the
original declares on its last layer is folded into `nn.CrossEntropyLoss` below instead, matching
every other classical head in this codebase.

Swap this call for any other name in `cnn.models.mcqcnn.VARIANTS` to turn this notebook into one
of the other nine.

In [12]:
net = CO_U1_QCNN(image_size=args.img_size,
                 n_input_channels=args.n_channels,
                 n_classes=args.n_classes).to(device)

print(net)
print('qubits per circuit:', net.qconv.n_qubits)
print('trainable parameters:', sum(p.numel() for p in net.parameters() if p.requires_grad))

MCQCNNModel(
  (qconv): U1Circuit()
  (relu): ReLU()
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=243, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=2, bias=True)
)
qubits per circuit: 5
trainable parameters: 7919


## Optimizer & Loss

Adam at `args.lr` and `nn.CrossEntropyLoss`, matching `train.py::train_model`'s
`model.compile(optimizer=Adam(learning_rate=global_learning_rate), loss='sparse_categorical_crossentropy')`
— PyTorch's `CrossEntropyLoss` takes integer class labels and applies the log-softmax itself, which
is exactly what Keras's *sparse* categorical cross-entropy does on top of the model's softmax.

In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=args.lr)

## Train

`train_mcqcnn` (from the handler) runs `args.epochs` passes over the training set — `model.fit`'s
loop — validating on the test set at the end of each epoch.

Expect this to be slow for the PCO variants: every step simulates a 13- or 15-qubit statevector for
`batch_size * 81 * 3` patches. The 4- and 5-qubit variants (CO, WEV, control) run in a fraction of
the time.

In [16]:
train_mcqcnn(net, optimizer, criterion, train_loader, test_loader, device,
             epochs=args.epochs)

epoch 1/5  step 5  loss: 0.6847
epoch 1/5  step 10  loss: 0.7615
epoch 1/5  step 15  loss: 0.8632
epoch 1/5  step 20  loss: 0.5886
epoch 1/5  step 25  loss: 0.4576
epoch 1/5  step 30  loss: 0.4015
epoch 1/5  step 35  loss: 0.7984
epoch 1/5  step 40  loss: 0.3700
epoch 1/5  step 45  loss: 0.3470
epoch 1/5  step 50  loss: 0.2117
epoch 1/5  step 55  loss: 0.5251
epoch 1/5  step 60  loss: 0.1735
epoch 1/5  step 65  loss: 0.8718
epoch 1/5  step 70  loss: 0.5030
epoch 1/5  step 75  loss: 0.8478
epoch 1/5  step 80  loss: 0.6015
epoch 1/5  step 85  loss: 0.4713
epoch 1/5  step 90  loss: 0.7994
epoch 1/5  step 95  loss: 0.7663
epoch 1/5  step 100  loss: 0.5553
epoch 1/5  step 105  loss: 0.7247
epoch 1/5  step 110  loss: 0.3680
epoch 1/5  step 115  loss: 0.4273
epoch 1/5  step 120  loss: 1.1212
epoch 1/5  step 125  loss: 0.5046
epoch 1/5  step 130  loss: 0.5985
epoch 1/5  step 135  loss: 0.4394
[epoch 1] train_loss: 0.5705  val_acc: 0.7308
epoch 2/5  step 5  loss: 0.3626
epoch 2/5  step 10  loss

## Evaluate all metrics

Same helper every notebook in this folder uses:
[`evaluate_all_metrics`](../../../_handlers/evaluation.py) runs the model once over one split and
prints **every** metric the training routines can produce — the medmnist Evaluator AUC/ACC plus
accuracy, weighted precision / recall (sensitivity) / F1, per-class + average specificity,
one-vs-rest AUC, the confusion matrix and a per-class report.

`MCQCNNLogits` adapts the model to the contract `evaluate_all_metrics` expects (raw images in,
logits out): it resizes each batch down to `img_size`, replicates the channels and feeds it
through. `size=28` tells the medmnist `Evaluator` to score against the native-resolution `.npz`
(the model never sees the `_224` variant). Set `split` to `'train'` or `'test'`.

In [17]:
split = 'test'   # 'train' or 'test'

eval_net = MCQCNNLogits(net, img_size=args.img_size, n_channels=args.n_channels)
eval_dataset = train_dataset if split == 'train' else test_dataset
metrics = evaluate_all_metrics(eval_net, eval_dataset, args.dataset, nb_classes=args.n_classes,
                               device=device, split=split, batch_size=args.batch_size, size=28)

100%|██████████| 39/39 [00:03<00:00, 11.72it/s]
[medmnist Evaluator]  auc: 0.7508  acc: 0.7372

=== test metrics (2 classes) ===
accuracy            : 0.7372
overall_accuracy    : 0.7372
auc (ovr)           : 0.7508
precision (weighted): 0.7118
recall / sensitivity: 0.7372
specificity (avg)   : 0.6096
f1 (weighted)       : 0.7167

per-class specificity: ['0.886', '0.333']

confusion matrix:
[[ 14  28]
 [ 13 101]]

classification report:
              precision    recall  f1-score   support

           0       0.52      0.33      0.41        42
           1       0.78      0.89      0.83       114

    accuracy                           0.74       156
   macro avg       0.65      0.61      0.62       156
weighted avg       0.71      0.74      0.72       156

